<a href="https://colab.research.google.com/github/strongman2026/jarred-ai-video/blob/main/colab_in.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import time

# ================= 1. GCS 挂载 =================
print("📁 正在挂载 GCS 永久云盘...")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/key.json"

# 安装 gcsfuse
!echo "deb https://packages.cloud.google.com/apt gcsfuse-focal main" | sudo tee /etc/apt/sources.list.d/gcsfuse.list > /dev/null
!curl -s https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add - > /dev/null
!apt-get -qq update && apt-get -qq install gcsfuse

!mkdir -p /content/my_gcs
# 挂载你的存储桶
!gcsfuse raleigh-comfyui-storage /content/my_gcs

# ================= 2. 初始化 ComfyUI (彻底清理版) =================
print("🚀 初始化 ComfyUI 环境...")
%cd /content
!rm -rf /content/ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI.git -q

# ================= 3. 资产映射 (防止重复链接报错) =================
print("🔗 链接 GCS 核心资产...")
# 在 GCS 中创建必要的持久化目录
!mkdir -p /content/my_gcs/ComfyUI_Data/custom_nodes
!mkdir -p /content/my_gcs/ComfyUI_Data/output
!mkdir -p /content/my_gcs/ComfyUI_Data/models/loras

# 移除 ComfyUI 自带的空文件夹
!rm -rf /content/ComfyUI/custom_nodes
!rm -rf /content/ComfyUI/output
!rm -rf /content/ComfyUI/models/loras

# 建立软链接
!ln -s /content/my_gcs/ComfyUI_Data/custom_nodes /content/ComfyUI/custom_nodes
!ln -s /content/my_gcs/ComfyUI_Data/output /content/ComfyUI/output
!ln -s /content/my_gcs/ComfyUI_Data/models/loras /content/ComfyUI/models/loras

# ================= 4. 安装插件 =================
%cd /content/ComfyUI/custom_nodes
![ ! -d "ComfyUI-Manager" ] && git clone https://github.com/ltdrdata/ComfyUI-Manager.git -q
![ ! -d "ComfyUI-WanVideoWrapper" ] && git clone https://github.com/kijai/ComfyUI-WanVideoWrapper.git -q

%cd /content/ComfyUI
!pip install -r requirements.txt -q
!pip install -r custom_nodes/ComfyUI-WanVideoWrapper/requirements.txt -q

# ================= 5. 拉取 Wan 2.1 14B I2V 模型 (V2V 专用，实测路径) =================
print("⚡ 正在从 HuggingFace 极速下载 Wan 2.1 14B I2V 旗舰模型...")
!pip install -U huggingface_hub -q
import huggingface_hub

# 下载 14B I2V (视频重绘最强模型)
try:
    huggingface_hub.snapshot_download(
        repo_id="Wan-AI/Wan2.1-I2V-14B-480P",
        allow_patterns=["models_i2v_14B/*"],
        local_dir="/content/ComfyUI/models/diffusion_models/Wan2.1-I2V-14B-480P",
        local_dir_use_symlinks=False
    )
    print("✅ 模型下载成功！")
except Exception as e:
    print(f"❌ 下载遇到点问题: {e}")

# ================= 6. Ngrok 启动 =================
print("🌐 唤醒 Ngrok 通道...")
!fuser -k 8188/tcp > /dev/null 2>&1
time.sleep(2)
!pip install pyngrok -q
from pyngrok import ngrok

NGROK_TOKEN = "394HSCYBOyi1ZXm5heUutThsefX_5W6E2JWnccTPNpmJauexK"
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()
public_url = ngrok.connect(8188).public_url

print("="*60)
print(f"👉 终极工作站链接: {public_url}")
print("="*60)

# 启动！
!python main.py

📁 正在挂载 GCS 永久云盘...
W: https://packages.cloud.google.com/apt/dists/gcsfuse-focal/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
{"timestamp":{"seconds":1773423041,"nanos":527176468},"severity":"INFO","message":"Start gcsfuse/3.7.3 (Go version go1.26.1) for app \"\" using mount point: /content/my_gcs\n","mount-id":"raleigh-comfyui-storage-92a302a2"}
{"timestamp":{"seconds":1773423041,"nanos":527198128},"severity":"INFO","message":"GCSFuse Config","mount-id":"raleigh-comfyui-storage-92a302a2","CLI Flags":{}}
{"timestamp":{"seconds":1773423041,"nanos":527216508},"severity":"INFO","message":"GCSFuse Config","mount-id":"raleigh-comfyui-storage-92a302a2","Full Config":{"AppName":"","CacheDir":"","CloudProfiler":{"Allocate

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 0 files: 0it [00:00, ?it/s]

✅ 模型下载成功！
🌐 唤醒 Ngrok 通道...
👉 终极工作站链接: https://carol-suppositionless-nonideologically.ngrok-free.dev
[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-03-13 17:30:59.959
** Platform: Linux
** Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
** Log path: /content/ComfyUI/user/comfyui.log

Prestartup times for custom nodes:
   3.5 seconds: /content/ComfyUI/custom_nodes/ComfyUI-Manager

Found comfy_kitchen backend cuda: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'scaled_mm_nvfp4']}
Found comf